# 1.2b Data preparation for generative AI

**Story:** SafeDrive wants a claims assistant that answers questions from the policy documents and
understands adjuster notes. Before any model sees this data you turn text and images into embeddings,
cut the policy documents into chunks, remove personal data from the notes, and build the training
files for model customization.

**Task statement 1.2:** Perform data transformation, feature engineering, and pre-processing (GenAI skills 1.2.5 to 1.2.9).

| Skill | Topic | Status |
|---|---|---|
| 1.2.5 | Embedding models for text and images | ✅ |
| 1.2.6 | Text pre-processing: tokenization, normalization, domain-specific augmentation | ✅ |
| 1.2.7 | Prepare documents for RAG: chunking strategies, metadata | ✅ |
| 1.2.8 | Mask, redact and anonymize data | ✅ |
| 1.2.9 | Datasets for fine-tuning, continued pre-training and distillation | ✅ |

💰 Cost of a full run: a few cents (a few hundred small Bedrock and Comprehend calls).

## Key points for the exam

- An **embedding** is a list of numbers that captures meaning; similar texts (or images) have vectors that are close (cosine similarity).
- **Titan Text Embeddings V2**: 256 / 512 / 1024 dimensions, up to 8,192 tokens. Fewer dimensions = cheaper storage and faster search, slightly lower accuracy.
  **Titan Multimodal Embeddings**: text and images in the **same** vector space (search images with text).
- Documents and queries must use the **same embedding model and dimension**. Changing the model = re-embed everything.
- **Tokens**, not characters, drive context limits and cost. Rough rule for English: 1 token ≈ 4 characters.
- **Chunking**: fixed-size (simple), semantic (split where the meaning changes), hierarchical (search small child chunks, return the larger parent). Add **metadata** for filtering.
- **PII**: Amazon Comprehend detects / redacts in text; Amazon Macie finds sensitive data in S3; Glue and DataBrew can detect and mask PII in tables.
- **Model customization data** (JSON Lines): fine-tuning = labelled prompt → answer pairs; continued pre-training = unlabelled text; distillation = prompts only (the teacher model writes the answers).

| Term | Meaning | Reversible? |
|---|---|---|
| Masking | Hide part of a value (`***-**-6789`) | No |
| Redaction | Remove the value or replace it with a tag (`[SSN]`) | No |
| Pseudonymization / tokenization | Replace with a stable token (hash, lookup) | Only with the key / table |
| Anonymization | Remove or generalize so that a person cannot be re-identified | No |

## Setup

In [ ]:
import sys; sys.path.append("..")
from common import utils as U, catchup as C
print(U.REGION, U.bucket())

Catch-up: the raw files (policy documents, notes, images, Q&A pairs) must be in S3.

In [ ]:
RAW = C.ensure_raw_data()
print(U.s3_uri(RAW))

Common imports, the Bedrock runtime client and two small JSON Lines helpers.

In [ ]:
import base64, io, json, os, re
import numpy as np
import pandas as pd
from common import rag

BUCKET = U.bucket()
GENAI = "safedrive/genai"
s3 = U.client("s3")
brt = U.client("bedrock-runtime")

def put_jsonl(rows, key):
    """Write a list of dicts to S3 as JSON Lines; returns the s3:// URI."""
    body = "\n".join(json.dumps(r) for r in rows) + "\n"
    s3.put_object(Bucket=BUCKET, Key=key, Body=body.encode())
    return U.s3_uri(key)

print(U.EMBED_MODEL, "|", U.IMAGE_EMBED_MODEL, "|", U.TEXT_MODEL)

## 1.2.5 Embedding models for text and images

- Text → vector with `InvokeModel` on an embedding model. Request and response are JSON.
- Titan V2 options: `dimensions` (256, 512, 1024) and `normalize` (unit length, so dot product = cosine similarity).
- For millions of documents use **Bedrock batch inference** instead of one call per text.

### Call the embedding model directly
Look at the response fields: the vector and the number of input tokens (this is what you pay for).

In [ ]:
body = {"inputText": "Is windscreen damage covered by my policy?", "dimensions": 256, "normalize": True}
resp = brt.invoke_model(modelId=U.EMBED_MODEL, body=json.dumps(body))
out = json.loads(resp["body"].read())
print("keys:", list(out))
print("length:", len(out["embedding"]), "| tokens:", out["inputTextTokenCount"], "| first values:", np.round(out["embedding"][:5], 3))

### Dimensions: 256, 512, 1024
The same text at three sizes. With `normalize=True` every vector has length 1.

In [ ]:
def embed(text, dim=1024):
    body = json.dumps({"inputText": text, "dimensions": dim, "normalize": True})
    return np.array(json.loads(brt.invoke_model(modelId=U.EMBED_MODEL, body=body)["body"].read())["embedding"])

for dim in (256, 512, 1024):
    v = embed("Is windscreen damage covered by my policy?", dim)
    print(dim, "-> shape", v.shape, "| norm", round(float(np.linalg.norm(v)), 3), "| float32 size", v.shape[0] * 4, "bytes")

### Similarity demo
One question and four candidate sentences. The glass sentence shares almost no words with the question
("windscreen" vs "glass") but should still score highest: embeddings capture meaning, not keywords.

In [ ]:
question = "My windscreen has a crack. Will SafeDrive pay for it?"
candidates = [
    "Glass cover: window and windshield repair has a reduced deductible of USD 50.",
    "Theft claims must be reported to the police within 24 hours.",
    "Monthly premium payment adds a 4% service fee.",
    "Preheat the oven to 180 degrees and bake the cake for 40 minutes.",
]
scores = {}
for dim in (256, 512, 1024):
    q = embed(question, dim)
    scores[dim] = [round(float(q @ embed(c, dim)), 3) for c in candidates]
pd.DataFrame(scores, index=[c[:45] for c in candidates])

The ranking is the same at every size; the scores move only a little. Pick the dimension by cost and scale:

| Dimension | Storage per vector (float32) | Choose when |
|---|---|---|
| 256 | 1 KB | Millions of vectors, cost and speed matter most |
| 512 | 2 KB | Middle ground |
| 1024 | 4 KB | Best retrieval accuracy (default; used for the SafeDrive Knowledge Base) |

### Multimodal embeddings: embed the damage photos
Titan Multimodal Embeddings takes a base64 image (and/or text) and returns one vector (256, 384 or 1024 long).

In [ ]:
def embed_mm(text=None, image_bytes=None, dim=384):
    body = {"embeddingConfig": {"outputEmbeddingLength": dim}}
    if text:
        body["inputText"] = text
    if image_bytes:
        body["inputImage"] = base64.b64encode(image_bytes).decode()
    resp = brt.invoke_model(modelId=U.IMAGE_EMBED_MODEL, body=json.dumps(body))
    v = np.array(json.loads(resp["body"].read())["embedding"])
    return v / np.linalg.norm(v)

manifest = pd.DataFrame(U.read_jsonl(f"{RAW}/images/manifest.jsonl"))
image_vecs = np.vstack([embed_mm(image_bytes=U.read_bytes(f"{RAW}/images/{f}")) for f in manifest["file"]])
print(image_vecs.shape)

### Search images with text
The text query is embedded by the same model into the same space, then compared with every image vector.
(The photos are simple drawings, so do not expect a perfect ranking.)

In [ ]:
query = "a red car"
idx, sims = rag.top_k(embed_mm(text=query), image_vecs, k=4)
result = manifest.iloc[idx][["file", "vehicle_color", "damage_type"]].copy()
result["similarity"] = sims.round(3)
print("query:", query)
result

### Search images with an image
Which photos look most like the first one? Position 0 is the photo itself (similarity 1.0).

In [ ]:
idx, sims = rag.top_k(image_vecs[0], image_vecs, k=4)
result = manifest.iloc[idx][["file", "vehicle_color", "damage_type"]].copy()
result["similarity"] = sims.round(3)
result

### Embedding types: float vs binary

| Type | One dimension is | 1024 dimensions take | Compare with |
|---|---|---|---|
| **Float** (default) | a 32-bit number | 4,096 bytes | cosine / dot product / Euclidean |
| **Binary** | one bit (0 or 1) | 128 bytes (**32x smaller**) | **Hamming distance** (count the bits that differ) |

- Titan Text Embeddings V2 returns either or both: `"embeddingTypes": ["float", "binary"]`.
- Binary vectors make huge indexes much cheaper and faster, with a small loss of accuracy.
- The vector store must support it (OpenSearch has a binary vector type).

Ask for both types in one call. The response has a new field, `embeddingsByType`.

In [ ]:
body = {"inputText": "Is windscreen damage covered by my policy?", "dimensions": 1024,
        "normalize": True, "embeddingTypes": ["float", "binary"]}
out = json.loads(brt.invoke_model(modelId=U.EMBED_MODEL, body=json.dumps(body))["body"].read())
by_type = out["embeddingsByType"]
print("types :", list(by_type))
print("float :", np.round(by_type["float"][:6], 3))
print("binary:", by_type["binary"][:16])

Storage: pack the bits into bytes and compare with float32.

In [ ]:
bits = np.array(by_type["binary"]) > 0
packed = np.packbits(bits)
print("float32:", len(by_type["float"]) * 4, "bytes | binary:", packed.nbytes, "bytes")

A helper that returns the bits for a text.

In [ ]:
def embed_binary(text, dim=1024):
    body = json.dumps({"inputText": text, "dimensions": dim, "embeddingTypes": ["binary"]})
    out = json.loads(brt.invoke_model(modelId=U.EMBED_MODEL, body=body)["body"].read())
    return np.array(out["embeddingsByType"]["binary"]) > 0

Rank the four candidate sentences both ways. Float: higher cosine = closer. Binary: **lower**
Hamming distance = closer. The order should be the same or nearly the same.

In [ ]:
type_question = "My windscreen has a crack. Will SafeDrive pay for it?"
q_float, q_bits = embed(type_question), embed_binary(type_question)
type_cmp = pd.DataFrame({
    "cosine (float)": [round(float(q_float @ embed(c)), 3) for c in candidates],
    "hamming (binary)": [int((q_bits != embed_binary(c)).sum()) for c in candidates],
}, index=[c[:45] for c in candidates])
type_cmp

> **Common pattern:** search a binary index first to get a shortlist (fast, cheap), then re-score
> the shortlist with the float vectors (accurate). Notebook 2.2b measures the recall of binary
> embeddings on the policy questions.

### Dense vs sparse embeddings

| | Dense | Sparse |
|---|---|---|
| Looks like | A few hundred to a few thousand numbers, all non-zero | One slot per vocabulary word, almost all zero |
| Captures | **Meaning** ("windscreen" is close to "glass") | **Exact words** (codes, names, rare terms) |
| Made by | Embedding model (Titan, Cohere, Nova) | TF-IDF / BM25 (counting), or a learned sparse model |
| Weak at | Exact IDs and rare tokens | Synonyms and paraphrases |
| Stored in | Vector index (HNSW ...) | Inverted index (what a search engine uses) |

**Hybrid search** = use both and merge the results.

Build sparse vectors for the candidate sentences with TF-IDF. Look at the shape and how many values are non-zero.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(stop_words="english")   # ignore words like "has", "the"
sparse_docs = tfidf.fit_transform(candidates)
print("shape:", sparse_docs.shape, "| non-zero values in sentence 0:", sparse_docs[0].nnz)

A sparse vector is readable: each non-zero slot is a word with a weight.

In [ ]:
row = sparse_docs[0].toarray()[0]
vocab = tfidf.get_feature_names_out()
sorted(((vocab[i], round(float(row[i]), 2)) for i in row.nonzero()[0]), key=lambda x: -x[1])[:6]

Two queries against the same sentences, scored with sparse and with dense vectors:
- Query A uses **different words** from the glass sentence ("windscreen", "crack").
- Query B repeats **exact words** from the service-fee sentence.

In [ ]:
def compare(query):
    sparse_scores = (sparse_docs @ tfidf.transform([query]).T).toarray().ravel()
    dense_scores = [float(embed(query) @ embed(c)) for c in candidates]
    return pd.DataFrame({"sparse": sparse_scores.round(3), "dense": np.round(dense_scores, 3)},
                        index=[c[:45] for c in candidates])

compare("My windscreen has a crack")

Sparse gives the glass sentence a score of 0: it shares no meaningful words with the query
("windscreen" vs "windshield" / "glass"). Dense should still rank it first.

In [ ]:
compare("4% service fee")

Here both should work, and sparse is very sure: the exact tokens match. With IDs or codes ("form SD-204")
sparse is usually the safer signal.

**Learned sparse embeddings** (for example OpenSearch *neural sparse* search) use a model to add
related words and weights to the sparse vector, so it also handles synonyms while staying an
inverted-index search. Notebook 1.1b runs keyword and hybrid search on OpenSearch; notebook 2.2b
measures sparse vs dense vs hybrid on the policy questions.

### Choosing an embedding model (notes)

| Model | Input | Remember |
|---|---|---|
| Amazon Titan Text Embeddings V2 | Text, 8K tokens, 100+ languages | Flexible dimensions; default for Bedrock Knowledge Bases |
| Amazon Titan Multimodal Embeddings G1 | Text + image | Image search, product / damage photo matching |
| Amazon Nova Multimodal Embeddings | Text, image, video, audio, documents | One space for many media types |
| Cohere Embed | Text (and images in v3 / v4) | `input_type` differs for documents and queries |
| SageMaker JumpStart embedding models | Text | When you must host the model yourself (VPC-only, custom model) |

## 1.2.6 Advanced text pre-processing

- **Tokenization**: split text into units. Classic NLP uses words; foundation models use **subword** tokens (BPE / SentencePiece).
- **Normalization**: make equal things look equal (case, whitespace, Unicode, abbreviations).
  Heavy cleaning (stop words, stemming) helps bag-of-words models; for FMs and embedding models keep the text natural.
- **Domain-specific augmentation**: create more training text that keeps the insurance meaning (paraphrase with an FM, swap domain synonyms, back-translation).

### Word tokens vs model tokens
Load the adjuster notes and tokenize one note by words. Then compare with the "4 characters per token" rule.

In [ ]:
notes = U.read_jsonl(f"{RAW}/adjuster_notes.jsonl")
note = notes[0]["note"]
words = re.findall(r"\w+|[^\w\s]", note)
print(note)
print("characters:", len(note), "| word tokens:", len(words), "| estimate chars/4:", len(note) // 4)
print(words[:14])

### Real token counts from the Converse API
Every Converse response has a `usage` block. `inputTokens` is the exact count the model billed for the prompt
(it includes a few tokens of message formatting).

In [ ]:
messages = [{"role": "user", "content": [{"text": note}]}]
resp = brt.converse(modelId=U.TEXT_MODEL, messages=messages, inferenceConfig={"maxTokens": 5})
print(resp["usage"])

### Count tokens without running the model
The `CountTokens` API returns the input token count for free, before you send a request. It is supported
only by some models, so the cell falls back to a message if your text model does not support it.

In [ ]:
try:
    counted = brt.count_tokens(modelId=U.TEXT_MODEL, input={"converse": {"messages": messages}})
    print("CountTokens:", counted["inputTokens"])
except Exception as e:  # noqa: BLE001
    print("CountTokens is not available for", U.TEXT_MODEL, "-", str(e)[:120])

### Why token counts matter: size up the corpus
Estimate tokens for every policy document (chars / 4). This drives the chunk size, the embedding cost and
whether a document fits in a context window.

In [ ]:
doc_keys = sorted(k for k in U.list_keys(f"{RAW}/policy_docs/") if k.endswith(".md"))
docs = {os.path.basename(k): U.read_bytes(k).decode() for k in doc_keys}
pd.DataFrame([{"doc": name, "chars": len(t), "words": len(t.split()), "est_tokens": len(t) // 4}
              for name, t in docs.items()])

### Text normalization
Unicode normalization, lower case, collapse whitespace and expand **domain abbreviations** that adjusters use.
Expanding abbreviations is a domain-specific step: "SIU" means nothing to a general model.

In [ ]:
import unicodedata

ABBREVIATIONS = {"siu": "special investigation unit", "ncb": "no-claim bonus",
                 "idv": "insured declared value", "tp": "third party"}

def normalize(text):
    text = unicodedata.normalize("NFKC", text).lower()
    text = re.sub(r"\s+", " ", text).strip()
    return re.sub(r"\b(" + "|".join(ABBREVIATIONS) + r")\b", lambda m: ABBREVIATIONS[m.group(1)], text)

sample = "Recommend  SIU review.\tCustomer asked about NCB and IDV  after the TP claim."
print(normalize(sample))

### Classic pipeline: stop words, stemming, TF-IDF
For a traditional text classifier, tokens become a sparse matrix of counts or TF-IDF weights.
Stemming cuts words to a root ("reported", "reporting" → "report").

In [ ]:
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer

stem = PorterStemmer().stem
print([stem(w) for w in ["reported", "reporting", "claims", "damaged", "collision"]])
tfidf = TfidfVectorizer(preprocessor=normalize, stop_words="english", ngram_range=(1, 2), min_df=5)
matrix = tfidf.fit_transform(n["note"] for n in notes)
print("matrix:", matrix.shape, "| sample terms:", list(tfidf.get_feature_names_out()[:12]))

### Domain-specific augmentation with a foundation model
Only a small share of notes belongs to fraud claims. We ask the FM to paraphrase one fraud note three ways,
keeping the facts and the insurance vocabulary. Check that names and numbers are unchanged.

In [ ]:
fraud_notes = [n for n in notes if n["is_fraud"] == 1]
print(len(fraud_notes), "fraud notes of", len(notes))
PROMPT = """You write insurance claim adjuster notes. Rewrite the note below in 3 different ways.
Keep every fact, name, number and policy ID exactly. Use the vocabulary of a motor claims adjuster.
Return only a JSON array of 3 strings.

Note: {note}"""
raw = U.converse(PROMPT.format(note=fraud_notes[0]["note"]), temperature=0.8)
print(raw)

### Parse and label the augmented rows
New rows copy the label of the original and are marked `augmented`, so they can be kept out of validation and test sets.

In [ ]:
def parse_json_list(text):
    try:
        return json.loads(text[text.find("["): text.rfind("]") + 1])
    except ValueError:
        return []

augmented = [{**fraud_notes[0], "note": t, "augmented": True} for t in parse_json_list(raw)]
pd.DataFrame([fraud_notes[0]] + augmented)[["claim_id", "is_fraud", "note"]]

### Augmentation without a model: domain synonym replacement
Cheap and fully controlled. A domain dictionary keeps the meaning correct (a general thesaurus would not know "SIU").

In [ ]:
import random

SYNONYMS = {"car": ["vehicle"], "customer": ["policyholder", "insured"], "theft": ["stolen vehicle"],
            "estimate": ["repair quote"], "SIU": ["Special Investigation Unit"]}

def synonym_augment(text, seed=0):
    rng = random.Random(seed)
    for word, options in SYNONYMS.items():
        text = re.sub(rf"\b{word}\b", lambda m: rng.choice(options), text)
    return text

print(fraud_notes[0]["note"])
print(synonym_augment(fraud_notes[0]["note"]))

## 1.2.7 Prepare documents for RAG

| Chunking strategy | How | Good / bad |
|---|---|---|
| Fixed size | N tokens or characters with overlap | Simple and predictable; can cut a sentence or rule in half |
| Semantic | Split where the meaning changes (headings, or embedding similarity between sentences) | Chunks are self-contained; more work |
| Hierarchical | Small child chunks for search, larger parent chunk returned to the model | Precise match **and** enough context |
| None | One chunk per file | For files that are already small (FAQs) |

Bedrock Knowledge Bases offers exactly these (default, fixed-size, hierarchical, semantic, none) plus a custom Lambda.
Here we do them by hand to see what they produce; notebook 3.2b lets the Knowledge Base do it.

### Fixed-size chunks with overlap
400 characters per chunk, 50 characters repeated between neighbours. Look at where chunk 1 ends: in the middle of a sentence.

In [ ]:
fixed = [c for name, text in docs.items() for c in rag.chunk_fixed(text, name, size=400, overlap=50)]
print(len(fixed), "chunks")
print("END OF CHUNK 0 ...", repr(fixed[0]["text"][-80:]))
print("START OF CHUNK 1 :", repr(fixed[1]["text"][:80]))

### Chunks by heading (structure-aware)
The policy documents are Markdown with `##` sections, and each section is one topic. One chunk per section,
with the document title added in front so the chunk makes sense on its own.

In [ ]:
by_heading = [c for name, text in docs.items() for c in rag.chunk_by_heading(text, name)]
print(len(by_heading), "chunks")
by_heading[5]

### Semantic chunks from embeddings: similarity between neighbouring sentences
When there are no headings, embed each sentence and compare it with the next one.
A **drop** in similarity marks a change of topic. We try it on one document.

In [ ]:
text = docs["03_claims_process.md"]
sentences = [s.strip() for s in re.split(r"(?<=[.!?])\s+|\n+", text) if len(s.strip()) > 25]
sent_vecs = np.vstack([embed(s, 256) for s in sentences])
neighbour_sim = (sent_vecs[:-1] * sent_vecs[1:]).sum(axis=1)
print(len(sentences), "sentences | similarity to next sentence:")
print(neighbour_sim.round(2))

### Semantic chunks: cut at the biggest drops
Break where the similarity is in the lowest 30%. Compare the breaks with the real section headings.

In [ ]:
threshold = np.percentile(neighbour_sim, 30)
semantic, current = [], [sentences[0]]
for sent, sim in zip(sentences[1:], neighbour_sim):
    if sim < threshold:
        semantic.append(" ".join(current))
        current = []
    current.append(sent)
semantic.append(" ".join(current))
print(len(semantic), "semantic chunks, threshold", round(float(threshold), 2))
for c in semantic[:4]:
    print("-", c[:110])

### Hierarchical chunks: parents and children
Parent = one section (from the heading chunks). Children = 200-character pieces of that section.
Each child remembers its `parent_id`.

In [ ]:
parents = [{**c, "parent_id": i} for i, c in enumerate(by_heading)]
children = [{"parent_id": p["parent_id"], "source": p["source"], "text": piece["text"]}
            for p in parents for piece in rag.chunk_fixed(p["text"], p["source"], size=200, overlap=20)]
print(len(parents), "parents,", len(children), "children")
children[0]

### Hierarchical retrieval: search children, return the parent
The small child matches the question precisely; the model then receives the whole parent section.

In [ ]:
child_vecs = np.vstack([embed(c["text"], 256) for c in children])
question = "How long can I have a rental car?"
idx, sims = rag.top_k(embed(question, 256), child_vecs, k=1)
best = children[idx[0]]
print("best child :", repr(best["text"][:120]))
print("its parent :", parents[best["parent_id"]]["text"][:300])

### Compare the strategies
Number of chunks and their sizes. More, smaller chunks = more precise search but more vectors to store.

In [ ]:
def stats(name, texts):
    lengths = [len(t) for t in texts]
    return {"strategy": name, "chunks": len(lengths), "avg_chars": int(np.mean(lengths)), "max_chars": max(lengths)}

pd.DataFrame([stats("fixed 400/50", [c["text"] for c in fixed]),
              stats("by heading", [c["text"] for c in by_heading]),
              stats("hierarchical children", [c["text"] for c in children])])

### Metadata: the sidecar files
Each policy document has a `<name>.metadata.json` file. A Bedrock Knowledge Base reads it during ingestion
and you can then **filter** searches (for example only `category = "claims"`).

In [ ]:
meta_rows = []
for name in docs:
    attrs = json.loads(U.read_bytes(f"{RAW}/policy_docs/{name}.metadata.json"))["metadataAttributes"]
    meta_rows.append({"source": name, **attrs})
doc_meta = pd.DataFrame(meta_rows)
doc_meta

### Metadata extraction with a foundation model
Extra metadata (keywords, a one-line summary) can be generated per chunk and stored next to it.
Here the FM extracts it for one chunk as JSON.

In [ ]:
chunk = by_heading[5]["text"]
prompt = f"""Extract metadata from this insurance policy text.
Return only JSON with the keys "summary" (one sentence) and "keywords" (3 to 5 short strings).

Text: {chunk}"""
print(U.converse(prompt, temperature=0))

### Build the final chunk table (reuse if it exists)
The assistant uses the **heading chunks** with 1024-dimension embeddings. This is the same table that
`C.ensure_chunks` builds, saved to the same place. If a later notebook already created it, we load it.

In [ ]:
CHUNKS_KEY = f"{GENAI}/chunks.parquet"
if U.get_state("chunks_key"):
    chunks_df = U.read_parquet(CHUNKS_KEY)
    print("reused", U.s3_uri(CHUNKS_KEY))
else:
    chunks_df = pd.DataFrame(by_heading)
    chunks_df["embedding"] = [U.embed(t) for t in chunks_df["text"]]
    print("embedded", len(chunks_df), "chunks")
chunks_df.drop(columns="embedding").head(3)

Save the table to S3 and record its key in the shared state.

In [ ]:
if not U.get_state("chunks_key"):
    U.write_df(chunks_df, CHUNKS_KEY, fmt="parquet")
    U.save_state(chunks_key=CHUNKS_KEY)
print(U.get_state("chunks_key"), "| columns:", list(chunks_df.columns))

### Test retrieval on the chunks
Embed a question, take the 3 nearest chunks. This is the "R" of RAG without any vector database.

In [ ]:
matrix = np.vstack(chunks_df["embedding"].to_numpy())
question = "What is the deductible for a driver aged 22?"
idx, sims = rag.top_k(U.embed(question), matrix, k=3)
hits = chunks_df.iloc[idx][["source", "heading"]].copy()
hits["similarity"] = sims.round(3)
hits

### Retrieval with a metadata filter
Join the document metadata to the chunks and search only inside one category.

In [ ]:
with_meta = chunks_df.merge(doc_meta, on="source", how="left")
mask = (with_meta["category"] == "claims").to_numpy()
idx, sims = rag.top_k(U.embed(question), matrix[mask], k=2)
hits = with_meta[mask].iloc[idx][["source", "heading", "category"]].copy()
hits["similarity"] = sims.round(3)
hits

## 1.2.8 Mask, redact and anonymize data

Adjuster notes contain names, phone numbers, e-mail addresses and partial SSNs. They must not end up in
prompts, embeddings or fine-tuning data.

| Tool | Use |
|---|---|
| **Amazon Comprehend** `DetectPiiEntities` / `ContainsPiiEntities` | Find PII in text (offsets + type); async jobs can write redacted copies |
| **Amazon Macie** | Discover and report sensitive data across S3 buckets (does not change the data) |
| **Glue Detect PII transform / DataBrew PII recipe steps** | Detect and mask, hash or encrypt columns in ETL |
| **Bedrock Guardrails** sensitive-information filters | Block or mask PII in prompts and answers at inference time |
| **Amazon Transcribe** PII redaction | Redact PII from call transcripts |

### Detect PII in one note
Each entity has a type, a confidence score and character offsets into the text.

In [ ]:
comp = U.client("comprehend")
note = next(n["note"] for n in notes if "SSN" in n["note"])
entities = comp.detect_pii_entities(Text=note, LanguageCode="en")["Entities"]
print(note)
pd.DataFrame([{**e, "text": note[e["BeginOffset"]:e["EndOffset"]], "Score": round(e["Score"], 3)} for e in entities])

### Redaction: replace each entity with its type
Work from the end of the text to the start so the offsets stay valid.

In [ ]:
def redact(text, entities, min_score=0.5):
    for e in sorted(entities, key=lambda e: e["BeginOffset"], reverse=True):
        if e["Score"] >= min_score:
            text = text[:e["BeginOffset"]] + f"[{e['Type']}]" + text[e["EndOffset"]:]
    return text

print(redact(note, entities))

### Masking: hide most of the value, keep the last 4 characters
Useful when staff still need to recognise a value ("the phone number ending in 9876").

In [ ]:
def mask(text, entities, keep=4):
    for e in sorted(entities, key=lambda e: e["BeginOffset"], reverse=True):
        value = text[e["BeginOffset"]:e["EndOffset"]]
        hidden = re.sub(r"\w", "*", value[:-keep]) + value[-keep:] if e["Type"] in ("PHONE", "SSN") else "*" * len(value)
        text = text[:e["BeginOffset"]] + hidden + text[e["EndOffset"]:]
    return text

print(mask(note, entities))

### Pseudonymization: replace with a stable token
A keyed hash (HMAC-SHA256) turns the same name into the same token every time, so records can still be
joined and counted, but the name cannot be read. Keep the key in AWS Secrets Manager or KMS; without it nobody can link tokens to people.

In [ ]:
import hashlib, hmac

SECRET = b"demo-key-store-me-in-secrets-manager"

def token(value, kind):
    return f"{kind}_{hmac.new(SECRET, value.lower().encode(), hashlib.sha256).hexdigest()[:10]}"

def pseudonymize(text, entities):
    for e in sorted(entities, key=lambda e: e["BeginOffset"], reverse=True):
        text = text[:e["BeginOffset"]] + token(text[e["BeginOffset"]:e["EndOffset"]], e["Type"]) + text[e["EndOffset"]:]
    return text

print(pseudonymize(note, entities))
print(token("John Smith", "NAME"), "==", token("john smith", "NAME"))

### Quick check: does a text contain PII at all?
`ContainsPiiEntities` returns only labels and scores (no offsets). Good as a cheap gate before text enters a pipeline.

In [ ]:
for text in [note, "Windscreen crack, simple glass claim, approved for cashless repair."]:
    labels = comp.contains_pii_entities(Text=text, LanguageCode="en")["Labels"]
    print([f"{l['Name']} {l['Score']:.2f}" for l in labels] or "no PII", "<-", text[:60])

### Redact a batch of notes and store the clean copy
40 notes are enough for the demo (Comprehend bills per 100 characters). The redacted file is what GenAI steps should read.

In [ ]:
redacted_rows = []
for n in notes[:40]:
    found = comp.detect_pii_entities(Text=n["note"], LanguageCode="en")["Entities"]
    redacted_rows.append({**n, "note": redact(n["note"], found), "pii_types": sorted({e["Type"] for e in found})})
print(put_jsonl(redacted_rows, f"{GENAI}/adjuster_notes_redacted.jsonl"))
pd.DataFrame(redacted_rows)[["claim_id", "pii_types", "note"]].head(4)

### Anonymize a table
For tabular data: **drop** direct identifiers (name), **pseudonymize** keys you still need for joins (policy ID)
and **generalize** quasi-identifiers (exact age → age band).

In [ ]:
policies = U.read_csv(f"{RAW}/policies.csv")
anon = policies.drop(columns=["customer_name"]).copy()
anon["policy_id"] = anon["policy_id"].map(lambda v: token(v, "POL"))
anon["age_band"] = pd.cut(anon["age"], bins=[17, 24, 39, 59, 120], labels=["18-24", "25-39", "40-59", "60+"])
anon = anon.drop(columns=["age"])
anon.head(3)

### k-anonymity check
Group by the quasi-identifiers. The smallest group size is *k*: every person is hidden among at least k people.
With exact age the groups are much smaller than with age bands.

In [ ]:
k_exact = policies.groupby(["age", "gender", "region"]).size().min()
k_banded = anon.groupby(["age_band", "gender", "region"], observed=True).size().min()
print("k with exact age:", k_exact, "| k with age bands:", k_banded)

### Managed options (notes)

```python
# Comprehend asynchronous job: writes redacted copies of every file under an S3 prefix
comp.start_pii_entities_detection_job(
    InputDataConfig={"S3Uri": "s3://bucket/notes/", "InputFormat": "ONE_DOC_PER_LINE"},
    OutputDataConfig={"S3Uri": "s3://bucket/notes-redacted/"},
    Mode="ONLY_REDACTION", LanguageCode="en", DataAccessRoleArn=role,
    RedactionConfig={"PiiEntityTypes": ["NAME", "PHONE", "EMAIL", "SSN"],
                     "MaskMode": "REPLACE_WITH_PII_ENTITY_TYPE"})
```

- **Macie**: enable it, run a sensitive-data discovery job on the bucket, review findings in Security Hub / EventBridge. It finds, it does not fix.
- **Glue Studio "Detect Sensitive Data"** transform: detect per cell or per column, then redact, hash or partially mask inside the ETL job.
- **DataBrew**: PII recipe steps (`REDACT`, hashing, encryption with KMS, shuffle, nullify).

## 1.2.9 Prepare data for fine-tuning, continued pre-training and distillation

| Customization | Data you provide | Format (JSON Lines, one record per line) | Goal |
|---|---|---|---|
| **Fine-tuning** | Labelled pairs (hundreds to thousands) | Nova / Claude: `{"schemaVersion": "bedrock-conversation-2024", "system": [...], "messages": [user, assistant]}`; Titan Text: `{"prompt": ..., "completion": ...}` | Teach a task or style |
| **Continued pre-training** | Unlabelled domain text (lots) | `{"input": "<text>"}` | Teach domain language and knowledge |
| **Distillation** | Prompts only (or invocation logs) | Conversation format with the user turn; the **teacher** model generates the answers | A small, cheap student that imitates a large teacher |

Rules that apply to all: UTF-8 JSONL in S3, one record per line, stay inside the model's record and token limits,
keep a separate validation file where the model supports it. The job itself is run in notebook 2.2b (🔒).

### Load the raw question-answer pairs
The file has good records and a few deliberately bad ones.

In [ ]:
qa_raw = U.read_jsonl(f"{RAW}/finetune_qa.jsonl")
print(len(qa_raw), "records")
qa_raw[0]

### Format validation
Structural checks only: the right keys, non-empty strings, and not too long.
(Duplicates, PII and toxic content are found in notebook 1.3, skill 1.3.6.)

In [ ]:
MAX_CHARS = 4000        # about 1,000 tokens per field

def check_record(r):
    if set(r) != {"prompt", "completion"}:
        return "wrong keys"
    if not all(isinstance(r[k], str) and r[k].strip() for k in r):
        return "empty field"
    if max(len(r["prompt"]), len(r["completion"])) > MAX_CHARS:
        return "too long"
    return "ok"

checked = pd.DataFrame({"reason": [check_record(r) for r in qa_raw]})
qa = [r for r in qa_raw if check_record(r) == "ok"]
print(len(qa), "records pass")
checked["reason"].value_counts()

### Fine-tuning file: conversation format (Amazon Nova)
Each record is one short conversation: optional system prompt, a user turn and the ideal assistant turn.

In [ ]:
SYSTEM = "You are the SafeDrive Insurance claims assistant. Answer briefly and only from SafeDrive policy."

def to_conversation(prompt, completion=None):
    messages = [{"role": "user", "content": [{"text": prompt}]}]
    if completion is not None:
        messages.append({"role": "assistant", "content": [{"text": completion}]})
    return {"schemaVersion": "bedrock-conversation-2024", "system": [{"text": SYSTEM}], "messages": messages}

ft_rows = [to_conversation(r["prompt"], r["completion"]) for r in qa]
print(json.dumps(ft_rows[0], indent=1))

### Validate the conversation records
Roles must alternate user → assistant, every turn needs text, and a fine-tuning record must end with the assistant.

In [ ]:
def check_conversation(rec, need_answer=True):
    if rec.get("schemaVersion") != "bedrock-conversation-2024":
        return "bad schemaVersion"
    roles = [m.get("role") for m in rec.get("messages", [])]
    expected = ["user", "assistant"] * (len(roles) // 2) + ["user"] * (len(roles) % 2)
    if not roles or roles != expected:
        return "roles must alternate, starting with user"
    if need_answer and roles[-1] != "assistant":
        return "last turn must be the assistant"
    if not all(m["content"] and m["content"][0].get("text", "").strip() for m in rec["messages"]):
        return "empty turn"
    return "ok"

print(pd.Series([check_conversation(r) for r in ft_rows]).value_counts().to_dict())
print("broken example ->", check_conversation({"schemaVersion": "bedrock-conversation-2024", "messages": ft_rows[0]["messages"][::-1]}))

### Fine-tuning file: prompt / completion format (Amazon Titan Text)
The older, simpler format. Same data, different shape. The right format depends on the base model you customize.

In [ ]:
pc_rows = [{"prompt": r["prompt"], "completion": r["completion"]} for r in qa]
print(json.dumps(pc_rows[0]))

### Continued pre-training file: unlabelled text
No questions, no answers: just domain text. We use one policy section per record.

In [ ]:
cpt_rows = [{"input": c["text"]} for c in by_heading]
print(len(cpt_rows), "records")
print(json.dumps(cpt_rows[0])[:230], "...")

### Distillation file: prompts only
The teacher model (for example Nova Pro) answers these prompts during the job and the student (Nova Micro / Lite)
learns from the answers. We combine the training questions with the evaluation questions.
Bedrock asks for at least 100 prompts; ours is a format demo.

In [ ]:
eval_questions = [r["question"] for r in U.read_jsonl(f"{RAW}/qa_eval.jsonl")]
prompts = sorted({r["prompt"] for r in qa} | set(eval_questions))
distill_rows = [to_conversation(p) for p in prompts]
print(len(distill_rows), "prompts |", {check_conversation(r, need_answer=False) for r in distill_rows})
print(json.dumps(distill_rows[0])[:260])

### Check the size limits
Typical Bedrock limits: a minimum number of records (8 for Nova fine-tuning, 100 prompts for distillation) and
a maximum token length per record. Always check the current quota page for your base model.

In [ ]:
def size_report(name, rows, minimum):
    chars = [len(json.dumps(r)) for r in rows]
    return {"file": name, "records": len(rows), "min_required": minimum, "enough": len(rows) >= minimum,
            "max_est_tokens": max(chars) // 4}

pd.DataFrame([size_report("fine-tuning", ft_rows, 8), size_report("continued pre-training", cpt_rows, 1),
              size_report("distillation", distill_rows, 100)])

### Upload the files
Everything goes to `s3://.../safedrive/genai/finetune/`. If notebook 1.3 already wrote a cleaned
`train.jsonl`, that file is kept and not overwritten.

In [ ]:
FT = f"{GENAI}/finetune"
uris = {
    "titan_style": put_jsonl(pc_rows, f"{FT}/train_prompt_completion.jsonl"),
    "continued_pretraining": put_jsonl(cpt_rows, f"{FT}/continued_pretraining.jsonl"),
    "distillation": put_jsonl(distill_rows, f"{FT}/distillation_prompts.jsonl"),
}
if not U.get_state("finetune_cleaned"):
    put_jsonl(ft_rows, f"{FT}/train.jsonl")
uris["fine_tuning"] = U.s3_uri(f"{FT}/train.jsonl")
uris

### Read one file back and validate every line
A last check on what is really in S3: every line must parse as JSON and pass the conversation check.

In [ ]:
back = U.read_jsonl(f"{FT}/train.jsonl")
print(len(back), "records |", pd.Series([check_conversation(r) for r in back]).value_counts().to_dict())

### How the files are used (note)
One API, three customization types. Hands-on (🔒, costs money) is in notebook 2.2b.

```python
bedrock.create_model_customization_job(
    jobName="safedrive-ft", customModelName="safedrive-nova-micro-ft", roleArn=role,
    baseModelIdentifier="amazon.nova-micro-v1:0:128k",
    customizationType="FINE_TUNING",            # or "CONTINUED_PRE_TRAINING" or "DISTILLATION"
    trainingDataConfig={"s3Uri": "s3://bucket/safedrive/genai/finetune/train.jsonl"},
    outputDataConfig={"s3Uri": "s3://bucket/safedrive/genai/finetune/output/"},
    hyperParameters={"epochCount": "2", "learningRate": "0.00001", "batchSize": "1"})
# Distillation adds customizationConfig={"distillationConfig": {"teacherModelConfig": {"teacherModelIdentifier": ...}}}
```

## Exam traps

- **Continued pre-training uses unlabelled data**; fine-tuning uses labelled pairs. "We have lots of raw documents and no labels" → continued pre-training (or RAG).
- **Distillation needs prompts, not answers**: the teacher generates the answers. Goal = lower cost and latency, not higher accuracy than the teacher.
- Query and documents must be embedded with the **same model and dimension**; you cannot mix Titan and Cohere vectors in one index.
- **Macie discovers** sensitive data in S3; it does not redact. Comprehend (text), Glue / DataBrew (tables) and Guardrails (inference) do the masking.
- Bigger chunks are not always better: they dilute the match and fill the context window. Hierarchical chunking gives precise match plus context.

## Quiz

**1.** A company has 40 GB of internal engineering manuals with no labels and wants a Bedrock model to
become familiar with its terminology. Which customization method fits the data?
- A. Fine-tuning with prompt-completion pairs
- B. Continued pre-training
- C. Model distillation
- D. Prompt caching

<details><summary>Answer</summary>B. Continued pre-training adapts a model with unlabelled domain text. Fine-tuning needs labelled pairs.</details>

**2.** A RAG application returns chunks that match the question well, but the answers lack surrounding context
because the chunks are very small. Which chunking strategy addresses this with the least loss of search precision?
- A. Fixed-size chunks of 8,000 tokens
- B. No chunking
- C. Hierarchical chunking (search child chunks, return parent chunks)
- D. Removing chunk overlap

<details><summary>Answer</summary>C. Small child chunks keep the match precise; the parent chunk supplies the context.</details>

**3.** Call-centre transcripts stored as text in S3 must have names and phone numbers replaced with the entity
type before they are used to fine-tune a model. Which service does this directly?
- A. Amazon Macie
- B. Amazon Comprehend PII detection with redaction
- C. Amazon Rekognition
- D. AWS Shield

<details><summary>Answer</summary>B. Comprehend detects PII entities in text and can write redacted output. Macie only discovers and reports.</details>

**4.** A vector index was built with Titan Text Embeddings V2 at 1024 dimensions. To save cost, a developer
starts embedding user queries at 256 dimensions. What happens?
- A. Search works but is slightly less accurate
- B. Queries fail or are meaningless because the dimensions do not match the index
- C. The index converts the vectors automatically
- D. Only the latency changes

<details><summary>Answer</summary>B. Query and document vectors must come from the same model with the same dimension; the index dimension is fixed.</details>

**5.** A team wants a small, low-latency model that answers like a much larger model on their support
questions. They have 5,000 real customer prompts but no reference answers. Which approach fits?
- A. Model distillation with the large model as teacher
- B. Continued pre-training of the large model
- C. Fine-tuning the small model on the prompts alone
- D. Increasing the temperature of the small model

<details><summary>Answer</summary>A. Distillation uses prompts only; the teacher writes the answers and the student model learns from them.</details>

## Save state
`chunks_key` was saved in section 1.2.7. Here we add the fine-tuning file that notebooks 1.3 and 2.2b use
(conversation format, `bedrock-conversation-2024`).

In [ ]:
state = U.save_state(finetune_train_s3=uris["fine_tuning"])
{k: state.get(k) for k in ("chunks_key", "finetune_train_s3")}

## Cleanup

This notebook created **no running resources**: only files in S3 under `safedrive/genai/`
(`chunks.parquet`, `adjuster_notes_redacted.jsonl`, `finetune/*.jsonl`). They are kept for later notebooks
and removed by `99_cleanup/99_teardown.ipynb`.

In [ ]:
for k in U.list_keys(GENAI + "/"):
    print(k)